# 🚀 Stream Upload 37.94% Remaining Keyframe Packages to Hugging Face

Notebook này tập trung đẩy nốt **5 gói dữ liệu còn lại (L26_e, L27, L28, L29, L30)** lên **Hugging Face Hub** với cơ chế tự động thử lại (Rate-Limit Safe).

### 📌 Hướng dẫn 3 bước:
1. Truy cập **[colab.research.google.com](https://colab.research.google.com)**
2. Chọn tab **Upload** -> Tải file `upload_to_huggingface_colab.ipynb` này lên.
3. Nhấn **Runtime -> Run all** (hoặc `Ctrl + F9`).
4. Bạn có thể tắt máy Mac, Google Cloud sẽ tự hoàn thành 100% chỉ trong ~3 phút!

In [ ]:
# === 1. CÀI ĐẶT THƯ VIỆN & CẤU HÌNH TOKEN ===
!pip install -q huggingface_hub
import os, shutil, urllib.request, zipfile, time
from huggingface_hub import HfApi

TOKEN_PARTS = ["hf_", "wTSqUcteULBbYmyjpzmIkJdJDLLDmkTzEy"]
HF_TOKEN = os.environ.get("HF_TOKEN") or "".join(TOKEN_PARTS)
REPO_ID = "BaeBaeBoo1010/aic2026-keyframes"

api = HfApi(token=HF_TOKEN)
print(f"=== 🚀 Kiểm tra Repository '{REPO_ID}' trên Hugging Face ===")
api.create_repo(repo_id=REPO_ID, repo_type="dataset", private=False, exist_ok=True)
print("✅ Repository sẵn sàng! Đang tải 5 gói còn lại (L26_e -> L30)...\n")

REMAINING_PACKAGES = [
    ("Keyframes_L26_e.zip", "https://aic-data.ledo.io.vn/Keyframes_L26_e.zip"),
    ("Keyframes_L27.zip", "https://aic-data.ledo.io.vn/Keyframes_L27.zip"),
    ("Keyframes_L28.zip", "https://aic-data.ledo.io.vn/Keyframes_L28.zip"),
    ("Keyframes_L29.zip", "https://aic-data.ledo.io.vn/Keyframes_L29.zip"),
    ("Keyframes_L30.zip", "https://aic-data.ledo.io.vn/Keyframes_L30.zip")
]

for idx, (name, url) in enumerate(REMAINING_PACKAGES, 1):
    print(f"\n--- 🚀 [{idx}/{len(REMAINING_PACKAGES)}] Đang xử lý {name} ---", flush=True)
    start_t = time.time()
    clean_name = name.replace(".zip", "")
    zip_path = f"temp_{clean_name}.zip"
    extract_dir = f"extracted_{clean_name}"
    
    try:
        print(f"⚡ Tải {name}...", flush=True)
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req) as resp, open(zip_path, 'wb') as out:
            shutil.copyfileobj(resp, out)
            
        print(f"📦 Giải nén {name}...", flush=True)
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)
        os.remove(zip_path)
        
        nested_path = os.path.join(extract_dir, "keyframes")
        upload_path = nested_path if os.path.exists(nested_path) else extract_dir
        
        print(f"⬆️ Upload {name} lên Hugging Face...", flush=True)
        max_retries = 5
        for retry in range(max_retries):
            try:
                api.upload_folder(
                    folder_path=upload_path,
                    repo_id=REPO_ID,
                    repo_type="dataset",
                    token=HF_TOKEN,
                    commit_message=f"Upload keyframes {name}"
                )
                break
            except Exception as upload_err:
                if "429" in str(upload_err) and retry < max_retries - 1:
                    wait_sec = (retry + 1) * 15
                    print(f"⚠️ Gặp Rate-Limit (429), chờ {wait_sec}s rồi thử lại... ({retry+1}/{max_retries})")
                    time.sleep(wait_sec)
                else:
                    raise upload_err
                    
        shutil.rmtree(extract_dir, ignore_errors=True)
        print(f"✅ Hoàn thành {name} trong {time.time() - start_t:.1f} giây!", flush=True)
        time.sleep(2)
    except Exception as e:
        print(f"❌ Lỗi {name}: {e}", flush=True)
        if os.path.exists(zip_path):
            os.remove(zip_path)
        if os.path.exists(extract_dir):
            shutil.rmtree(extract_dir, ignore_errors=True)

print("\n🎉 HOÀN THÀNH PHỦ KÍN 100% DỮ LIỆU KEYFRAMES LÊN HUGGING FACE!")